In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# directories

#/kaggle/input/herbarium-2022-fgvc9/train_metadata.json
#/kaggle/input/herbarium-2022-fgvc9/sample_submission.csv
#/kaggle/input/herbarium-2022-fgvc9/test_metadata.json
#/kaggle/input/herbarium-2022-fgvc9/train_images/135/47/13547__018.jpg

In [ ]:
import matplotlib.pyplot as plt

## 1. Load metadata 

In [ ]:
main_dir = '/kaggle/input/herbarium-2022-fgvc9'
metadata_file = os.path.join(main_dir, "{}_metadata.json")
images_dir = os.path.join(main_dir, "{}_images")
metadata_file, images_dir

In [ ]:
train_prefix = 'train'
test_prefix = 'test'

In [ ]:
#load metadata
import json

def get_dataframe(prefix):
    with open(metadata_file.format(prefix)) as f:
        metadata = json.load(f)
    
    if (prefix == train_prefix):
        df_ann = pd.DataFrame(metadata['annotations'])
        df_ann = df_ann[['image_id', 'category_id']]
        df_ann.set_index('image_id', inplace = True)

        df_images = pd.DataFrame(metadata['images'])
        df_images = df_images[['image_id', 'file_name']]
        df_images.set_index('image_id', inplace = True)

        return df_ann.join(df_images, how = 'left')
    else:
        df = pd.DataFrame(metadata)
        return df

In [ ]:
train_df = get_dataframe(train_prefix)
print(len(train_df))
train_df.head()

In [ ]:
test_df = get_dataframe(test_prefix)
print(len(test_df))
test_df.tail()

In [ ]:
train_df.isna().sum()

In [ ]:
test_df.isna().sum()

In [ ]:
train_df.category_id.hist(figsize = (25, 5))

In [ ]:
# how many examples of each class are presented in train dataset?
class_count = {target: len(train_df[train_df['category_id'] == target]) for target in sorted(train_df.category_id.unique())}
min(class_count.values()), max(class_count.values())

In [ ]:
import seaborn as sns

plt.figure(figsize = (15,5))
sns.countplot(x = list(class_count.values()))

In [ ]:
# a good approach to solve this task is to increase number of less presented classes by using image random transformations
# but we would get a huuuuge dataset and it would take a lot of time to process all these images that we cannot afford in case of limited resources
# so instead of this we take only frequently encounted class samples

# let's say that the minimum number of sample of one class should be 80 or more
threshold = 70
class_count = {k:v for k, v in class_count.items() if v >= threshold}

print(len(class_count))
plt.figure(figsize = (15,5))
sns.countplot(x = list(class_count.values()))

In [ ]:
train_df = pd.concat([train_df[train_df['category_id'] == target] for target in class_count.keys()])
print(len(train_df))

In [ ]:
import matplotlib.image as mpimg

image = mpimg.imread(os.path.join(images_dir.format(train_prefix), train_df.iloc[0]['file_name']))

plt.figure(figsize = (5,10))
plt.imshow(image)
plt.show()

## 2. Download images

In [ ]:
# preprocess label value
from sklearn import preprocessing

le = preprocessing.LabelEncoder()
train_df['category_id_encoded'] = le.fit_transform(train_df['category_id'])
train_df.head()

In [ ]:
# how many classes we have in the end?
num_of_classes = max(train_df['category_id_encoded']) + 1
num_of_classes

In [ ]:
# ImageDataGenerator().flow_from_dataframe(class_mode = 'sparse') requires string labels
train_df['category_id_encoded'] = train_df['category_id_encoded'].astype('str')
train_df = train_df.sample(frac = 1)
train_df

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 120
train_datagen = ImageDataGenerator(rescale = 1/255.,
                                   validation_split = 0.1)

In [ ]:
train_dir = images_dir.format(train_prefix)
print(train_dir)

train_gen = train_datagen.flow_from_dataframe(dataframe = train_df,
                                              directory = train_dir,
                                              x_col = "file_name",
                                              y_col = "category_id_encoded",
                                              target_size=(img_size, img_size),
                                              batch_size = 16,
                                              class_mode = 'sparse',
                                              subset = 'training')

val_gen = train_datagen.flow_from_dataframe(dataframe = train_df,
                                            directory = train_dir,
                                            x_col = "file_name",
                                            y_col = "category_id_encoded",
                                            target_size = (img_size, img_size),
                                            batch_size = 16,
                                            class_mode = 'sparse',
                                            subset = 'validation')

## 3. Build a model

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, Add
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

input_size = (img_size, img_size, 3) # all images are 120x120 size + RGB

input = Input(input_size)

out = Conv2D(16, 3)(input)
out = MaxPooling2D()(out)
out = BatchNormalization()(out)

out = Conv2D(32, 3)(out)
out = Conv2D(32, 3)(out)
out = MaxPooling2D()(out)
out = BatchNormalization()(out)

out = Conv2D(64, 3)(out)
out = MaxPooling2D()(out)
out = BatchNormalization()(out)

out = Flatten()(out)
out = Dropout(0.3)(out)
out = Dense(num_of_classes)(out)
out = Dropout(0.3)(out)
out = Dense(num_of_classes, activation = 'softmax')(out)

model = Model(inputs = input, outputs = out)

opt = Adam(learning_rate = 1e-3)

model.compile(optimizer = opt, loss = 'sparse_categorical_crossentropy')
model.summary()

In [ ]:
history = model.fit(train_gen, validation_data = val_gen, epochs = 3, verbose = 1)

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)

plt.plot(epochs, loss, 'bo', label = 'Trainig loss')
plt.plot(epochs, val_loss, 'r', label = 'Validation loss')
plt.legend()
plt.show()

In [ ]:
pred = []
y_true = []
for i in range(len(val_gen)):
    X_batch, y_batch = val_gen[i]
    pred.extend([np.argmax(x) for x in model.predict(X_batch, verbose = 0)])
    y_true.extend(y_batch)
    
print(sum(np.array(pred) == np.array(y_true))/len(y_true))

## 4. Predict

In [ ]:
test_dir = images_dir.format(test_prefix)
test_datagen = ImageDataGenerator(rescale = 1./255)
test_gen = test_datagen.flow_from_dataframe(dataframe = test_df, 
                                            directory = test_dir,
                                            x_col = "file_name",
                                            target_size = (img_size, img_size),
                                            batch_size = 32,
                                            class_mode = None,
                                            shuffle = False)

In [ ]:
pred = []
for i in range(len(test_gen)):
    pred.extend([np.argmax(x) for x in model.predict(test_gen[i], verbose = 0)])

In [ ]:
pred = le.inverse_transform(pred)

In [ ]:
sub = pd.read_csv(os.path.join(main_dir, 'sample_submission.csv'))
sub['Predicted'] = pred[:len(test_df)]
sub.head()

In [ ]:
sub.to_csv('submission.csv', index = False)  